In [4]:
import operator
import random
import json
from deap import base, creator, tools, gp
import pydot

# --------------------------------------------------------------------------
# 1. 기본 설정 (✨ 다양성 증가)
# --------------------------------------------------------------------------
# ✨ 지표 추가: MACD, ATR, BB 추가
INDICATORS = ["RSI", "SMA", "MACD", "ATR", "BB"]
# ✨ 파라미터 추가
NUM_INDICATOR_PARAMS = [5, 10, 14, 20, 30, 50, 100, 150, 200]
SOURCES = ["open", "high", "low", "close", "volume"]
OPS = [operator.gt, operator.ge, operator.eq, operator.ne, operator.lt, operator.le]

class BuyType(object): pass
class SellType(object): pass

def Strategy(buy_logic, sell_logic):
    return (buy_logic, sell_logic)

class ConditionManager:
    # ✨ 초기 조건 풀 확장: 20 -> 50
    def __init__(self, num_conditions=50):
        self.buy_pool = {}
        self.sell_pool = {}
        self.generate_conditions(num_conditions)
    
    def _create_random_condition(self):
        indicator = random.choice(INDICATORS)
        source = random.choice(SOURCES)
        param = random.choice(NUM_INDICATOR_PARAMS)
        op_func = random.choice(OPS)
        value = random.uniform(1, 100)
        return {"indicator": indicator, "source": source, "param": param, "op": op_func.__name__, "value": round(value, 2)}

    def generate_conditions(self, n):
        for i in range(n):
            self.buy_pool[f"buy_system{i + 1}"] = self._create_random_condition()
            self.sell_pool[f"sell_system{i + 1}"] = self._create_random_condition()

# --------------------------------------------------------------------------
# 2. DEAP 환경 설정
# --------------------------------------------------------------------------
condition_manager = ConditionManager()

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

pset = gp.PrimitiveSetTyped("Main", [], object)
pset.addPrimitive(Strategy, [BuyType, SellType], object)
pset.addPrimitive(operator.and_, [BuyType, BuyType], BuyType)
pset.addPrimitive(operator.or_, [BuyType, BuyType], BuyType)
pset.addPrimitive(operator.not_, [BuyType], BuyType)
pset.addPrimitive(operator.and_, [SellType, SellType], SellType)
pset.addPrimitive(operator.or_, [SellType, SellType], SellType)
pset.addPrimitive(operator.not_, [SellType], SellType)

for term in condition_manager.buy_pool.keys():
    pset.addTerminal(term, BuyType) 
for term in condition_manager.sell_pool.keys():
    pset.addTerminal(term, SellType)

# --------------------------------------------------------------------------
# 3. 커스텀 교차 & 변이 함수
# --------------------------------------------------------------------------
def custom_crossover(ind1, ind2):
    if len(ind1) < 2 or len(ind2) < 2 or ind1[0].name != 'Strategy' or ind2[0].name != 'Strategy':
        return ind1, ind2

    if random.random() < 0.5:
        slice1 = ind1.searchSubtree(1)
        slice2 = ind2.searchSubtree(1)
        sub_tree1, sub_tree2 = gp.PrimitiveTree(ind1[slice1]), gp.PrimitiveTree(ind2[slice2])
        gp.cxOnePoint(sub_tree1, sub_tree2)
        ind1[slice1], ind2[slice2] = sub_tree1, sub_tree2
    else:
        buy_slice1, buy_slice2 = ind1.searchSubtree(1), ind2.searchSubtree(1)
        sub_tree1, sub_tree2 = gp.PrimitiveTree(ind1[buy_slice1.stop:]), gp.PrimitiveTree(ind2[buy_slice2.stop:])
        gp.cxOnePoint(sub_tree1, sub_tree2)
        ind1[buy_slice1.stop:], ind2[buy_slice2.stop:] = sub_tree1, sub_tree2
            
    return ind1, ind2

def custom_mutation(individual):
    if len(individual) < 2 or individual[0].name != 'Strategy':
        return individual,

    if random.random() < 0.5:
        subtree_slice = individual.searchSubtree(1)
    else:
        buy_subtree_slice = individual.searchSubtree(1)
        subtree_slice = slice(buy_subtree_slice.stop, len(individual))

    selected_subtree_list = individual[subtree_slice]
    if not selected_subtree_list: return individual,

    temp_tree = gp.PrimitiveTree(selected_subtree_list)
    index = random.randrange(len(temp_tree))
    node = temp_tree[index]
    
    # ✨ 변이 시 생성되는 하위 트리 최대 깊이 증가: 3 -> 4
    new_expr = gp.genHalfAndHalf(pset, 1, 4, type_=node.ret)
    
    node_slice = temp_tree.searchSubtree(index)
    temp_tree[node_slice] = new_expr
    
    individual[subtree_slice] = temp_tree
    
    return individual,

# --------------------------------------------------------------------------
# 4. 평가, 파싱, 시각화 함수
# --------------------------------------------------------------------------
def eval_func(individual):
    if not individual or individual[0].name != 'Strategy':
        return -1000.0,
    # ✨ 트리 크기 제약 완화: 50 -> 100
    if len(individual) > 100:
        return -100.0,
    return random.uniform(10, 100),

def parse_gp_tree_to_json(individual, condition_manager):
    if not isinstance(individual, gp.PrimitiveTree) or individual[0].name != 'Strategy':
        return None

    buy_subtree_slice = individual.searchSubtree(1)
    buy_tree = gp.PrimitiveTree(individual[buy_subtree_slice])
    sell_tree = gp.PrimitiveTree(individual[buy_subtree_slice.stop:])

    buy_op_str = str(buy_tree) if len(buy_tree) > 0 else "NO_LOGIC"
    sell_op_str = str(sell_tree) if len(sell_tree) > 0 else "NO_LOGIC"

    buy_systems_obj = {node.name: condition_manager.buy_pool[node.name] for node in buy_tree if isinstance(node, gp.Terminal)}
    sell_systems_obj = {node.name: condition_manager.sell_pool[node.name] for node in sell_tree if isinstance(node, gp.Terminal)}

    dynamic_vars = {}
    for system_details in {**buy_systems_obj, **sell_systems_obj}.values():
        var_key = f"{system_details['indicator']}_{system_details['param']}"
        dynamic_vars[var_key] = system_details['param']

    return {
        "vars": dynamic_vars, "buy_systems": buy_systems_obj, "buy_system_op": buy_op_str,
        "sell_systems": sell_systems_obj, "sell_system_op": sell_op_str
    }

def visualize_tree(individual, filename="best_strategy_tree.png"):
    nodes, edges, labels = gp.graph(individual)
    graph = pydot.Dot(graph_type='graph', bgcolor="#f0f0f0") 
    
    for node_idx, node_label in labels.items():
        if node_label == 'Strategy': fillcolor = "#c5e3c5" # 녹색
        elif node_label.startswith("buy_system"): fillcolor = "#f2c5c5" # 붉은색
        elif node_label.startswith("sell_system"): fillcolor = "#c5d5e3" # 파란색
        elif node_label in ['and_', 'or_', 'not_']: fillcolor = "#e3d1c5" # 갈색
        else: fillcolor = "#ffffff"
        graph.add_node(pydot.Node(node_idx, label=node_label, style="filled", fillcolor=fillcolor))

    for edge in edges:
        graph.add_edge(pydot.Edge(edge[0], edge[1]))

    try:
        graph.write_png(filename)
        print(f"\n✅ 트리 시각화 완료: '{filename}' 파일이 저장되었습니다.")
    except OSError as e:
        print(f"\n❌ 트리 시각화 실패: Graphviz를 찾을 수 없습니다.")
        print("   Graphviz 프로그램을 설치하고 시스템 PATH에 등록했는지 확인하세요.")
        print(f"   (오류 메시지: {e})")

# --------------------------------------------------------------------------
# 5. Toolbox 설정 및 메인 실행 블록
# --------------------------------------------------------------------------
toolbox = base.Toolbox()
# ✨ 초기 트리 최대 깊이 증가: 4 -> 6
toolbox.register("expr_init", gp.genHalfAndHalf, pset=pset, min_=2, max_=6, type_=object)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr_init)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", eval_func)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("mate", custom_crossover)
toolbox.register("mutate", custom_mutation)

if __name__ == "__main__":
    random.seed(42)
    
    # ✨ 진화 환경 조정 (더 많은 탐색)
    N_POP = 100      # 개체군 크기: 50 -> 100
    N_GEN = 20      # 세대 수: 10 -> 20
    CXPB = 0.8      
    MUTPB = 0.15    

    pop = toolbox.population(n=N_POP)
    
    print("✨ 초기 개체군 생성 및 평가 시작...")
    fitnesses = map(toolbox.evaluate, pop)
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit
    
    print("="*60)
    print("🚀 유전 프로그래밍 진화 시작!")
    print(f"세대 수: {N_GEN}, 개체군 크기: {N_POP}, 교차/변이 확률: {CXPB}/{MUTPB}")
    print("="*60)

    for g in range(N_GEN):
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        pop[:] = offspring

        fits = [ind.fitness.values[0] for ind in pop]
        mean = sum(fits) / len(pop)
        
        print(f"> 세대 {g+1:02d}: 최고={max(fits):.2f}, 평균={mean:.2f}")

    print("="*60)
    print("🏆 최종 최적 전략 탐색 완료!")
    
    best_ind = tools.selBest(pop, 1)[0]
    final_json_strategy = parse_gp_tree_to_json(best_ind, condition_manager)
    
    print(f"\n최적 개체 트리 구조 (크기: {len(best_ind)}):\n{str(best_ind)}")
    print(f"\n최고 적합도: {best_ind.fitness.values[0]:.2f}")
    
    if final_json_strategy:
        print("\n[ 최종 JSON 출력 ]")
        print(json.dumps(final_json_strategy, indent=4))
        visualize_tree(best_ind)
    else:
        print("\n❌ 최종 JSON 생성 실패: 최적 개체가 유효한 'Strategy' 구조가 아닙니다.")

✨ 초기 개체군 생성 및 평가 시작...
🚀 유전 프로그래밍 진화 시작!
세대 수: 20, 개체군 크기: 100, 교차/변이 확률: 0.8/0.15
> 세대 01: 최고=96.24, 평균=-799.74
> 세대 02: 최고=98.71, 평균=-516.90
> 세대 03: 최고=99.36, 평균=-157.75
> 세대 04: 최고=99.77, 평균=58.55
> 세대 05: 최고=99.35, 평균=54.96
> 세대 06: 최고=99.41, 평균=55.10
> 세대 07: 최고=99.41, 평균=59.25
> 세대 08: 최고=99.41, 평균=57.82
> 세대 09: 최고=97.68, 평균=57.53
> 세대 10: 최고=98.37, 평균=60.33
> 세대 11: 최고=99.67, 평균=65.58
> 세대 12: 최고=99.67, 평균=58.23
> 세대 13: 최고=99.12, 평균=59.07
> 세대 14: 최고=99.89, 평균=62.05
> 세대 15: 최고=99.18, 평균=61.21
> 세대 16: 최고=98.30, 평균=59.07
> 세대 17: 최고=99.76, 평균=58.44
> 세대 18: 최고=99.37, 평균=57.45
> 세대 19: 최고=96.52, 평균=54.63
> 세대 20: 최고=98.78, 평균=61.12
🏆 최종 최적 전략 탐색 완료!

최적 개체 트리 구조 (크기: 24):
Strategy(not_(and_('buy_system13', not_(not_(or_('buy_system3', 'buy_system36'))))), and_(or_(and_('sell_system13', 'sell_system20'), or_('sell_system40', 'sell_system49')), or_('sell_system49', and_(or_('sell_system13', 'sell_system22'), 'sell_system50'))))

최고 적합도: 98.78

[ 최종 JSON 출력 ]
{
    "vars": {
    

/home/tako/Documents/yonghan/GenTradingRuleWithGP/.conda/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/tako/Documents/yonghan/GenTradingRuleWithGP/.conda/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
